<a href="https://colab.research.google.com/github/pskarthikk/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks, my rule, and its reason codes

### Signal checks

**1. Staleness — MIXED**

The declining rate rises from 51.14% for pages updated within 30 days to 61.11% for pages 91–180 days old. However, the 181+ bucket falls to 47.13% and contains only 174 pages. I therefore treat staleness as a directional signal rather than a proven predictor.

**2. Search visibility (`impressions_90d`) — MIXED**

The lowest-impression quartile has a 37.61% declining rate, while Q2 and Q3 are 60.46% and 62.56%. The highest quartile falls to 56.20%, so the relationship is not monotonic. I use impressions as an opportunity/visibility signal, not as a claim that high impressions alone cause decline.

### Baseline rule

Prioritize a page for refresh review when it has **at least 91 days since its last update** and **at least median 90-day search impressions (731)**.

The score combines a simple staleness level with existing search visibility. No fitted weights are used.

### Reason code

- `stale_and_visible` — page is at least 91 days old since its last update and has at least 731 impressions in the trailing 90 days.
- `not_priority` — page does not meet both baseline conditions.

### Action labels

- `refresh_review` — prioritize for editorial review.
- `monitor` — do not prioritize under this baseline.

In [3]:
!git clone https://github.com/pskarthikk/flyrank-ml-internship.git
%cd flyrank-ml-internship

!ls

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 137 (delta 49), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.87 MiB | 15.68 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/flyrank-ml-internship
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [2]:
# Inspect the available files and folders needed for ML-07
from pathlib import Path

print("Current directory:", Path.cwd())
print("\nwork/notebooks exists:", Path("work/notebooks").exists())
print("data/processed exists:", Path("data/processed").exists())
print("data/raw exists:", Path("data/raw").exists())

if Path("data/processed").exists():
    print("\nProcessed files:")
    for p in Path("data/processed").iterdir():
        print(" -", p)

if Path("data/raw").exists():
    print("\nRaw files:")
    for p in Path("data/raw").iterdir():
        print(" -", p)

Current directory: /content

work/notebooks exists: False
data/processed exists: False
data/raw exists: False


In [4]:
from pathlib import Path

print("Available skills:")
for p in Path("skills").rglob("README.md"):
    print(p)

print("\nSkill directories:")
for p in Path("skills").iterdir():
    print("-", p)

Available skills:
skills/README.md

Skill directories:
- skills/building-baselines
- skills/deploying-static-pages
- skills/writing-data-contracts
- skills/README.md
- skills/writing-research-papers
- skills/flyrank
- skills/framing-ml-problems
- skills/hunting-leakage-and-validating
- skills/directing-your-ai-assistant
- skills/writing-honest-claims
- skills/querying-big-datasets
- skills/auditing-signals
- skills/training-honest-models


In [5]:
from pathlib import Path

files = [
    Path("skills/building-baselines/SKILL.md"),
    Path("skills/flyrank/flyrank-data/SKILL.md"),
]

for file in files:
    print("\n" + "=" * 80)
    print(file)
    print("=" * 80)
    print(file.read_text())


skills/building-baselines/SKILL.md
---
name: building-baselines
description: Builds the transparent rule-based baseline every model must beat — a hand-written score with reason codes, ranked output, and precision@K evaluation. Use before training any model, or when someone reports model results with nothing to compare against.
---

# Building baselines

A model without a baseline is a number without a meaning. The baseline is a rule a human can
read — and its job is to be honestly beatable.

## Build it in this order

**1. Say the rule in plain words first.** "A page is worth reviewing if it used to get traffic,
it's getting old, and its position is slipping." If you can't say it, you can't code it.

**2. Code it as a transparent score.** Multiply/add simple conditions; no fitted weights:

```python
stale   = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions"]     # readable on purpose
```

**3

In [6]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
for col in df.columns:
    print("-", col)

Rows: 30000
Columns: 44

Column names:
- content_id
- client_id
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier
- trend_direction
- trend_pct


In [7]:
# Inspect the two candidate signals
signals = [
    "days_since_last_update",
    "impressions_90d"
]

print("Signal summary:\n")
print(df[signals].describe())

print("\nMissing values:\n")
print(df[signals].isna().sum())

print("\nStaleness values:\n")
print(df["days_since_last_update"].sort_values().head(10).to_list())
print("...")
print(df["days_since_last_update"].sort_values(ascending=False).head(10).to_list())

Signal summary:

       days_since_last_update  impressions_90d
count            30000.000000     30000.000000
mean                46.098300      5200.366300
std                 42.078709     16838.019547
min                  1.000000         1.000000
25%                 20.000000        81.000000
50%                 20.000000       731.000000
75%                104.000000      3615.250000
max                373.000000    517715.000000

Missing values:

days_since_last_update    0
impressions_90d           0
dtype: int64

Staleness values:

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
...
[373, 373, 373, 372, 372, 335, 334, 334, 313, 313]


In [8]:
print("trend_direction values:")
print(df["trend_direction"].value_counts(dropna=False))

print("\ntrend_pct summary:")
print(df["trend_pct"].describe())

print("\nOther potentially useful categorical signals:")
print("\ncontent_type:")
print(df["content_type"].value_counts(dropna=False))

print("\nfreshness_tier:")
print(df["freshness_tier"].value_counts(dropna=False))

trend_direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_pct summary:
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Other potentially useful categorical signals:

content_type:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

freshness_tier:
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64


In [9]:
# ML-07 Section 1 — Signal audit
# We use the observed "down" outcome ONLY to audit the signals.
# It will NOT be used as an input to the final scoring rule.

df["audit_declining"] = (df["trend_direction"] == "down").astype(int)

# ---------------------------------------------------------
# Signal 1: Staleness
# ---------------------------------------------------------
stale_bins = [-1, 30, 90, 180, float("inf")]
stale_labels = ["0-30", "31-90", "91-180", "181+"]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels
)

stale_audit = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("audit_declining", "size"),
          declining_rate=("audit_declining", "mean")
      )
      .reset_index()
)

print("SIGNAL 1 — STALENESS")
print(stale_audit.to_string(index=False))
print()

# ---------------------------------------------------------
# Signal 2: Search visibility
# ---------------------------------------------------------
df["impressions_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    labels=["Q1_lowest", "Q2", "Q3", "Q4_highest"]
)

impressions_audit = (
    df.groupby("impressions_bucket", observed=False)
      .agg(
          n=("audit_declining", "size"),
          declining_rate=("audit_declining", "mean")
      )
      .reset_index()
)

print("SIGNAL 2 — IMPRESSIONS_90D")
print(impressions_audit.to_string(index=False))

SIGNAL 1 — STALENESS
staleness_bucket     n  declining_rate
            0-30 20480        0.511377
           31-90   175        0.588571
          91-180  9171        0.611057
            181+   174        0.471264

SIGNAL 2 — IMPRESSIONS_90D
impressions_bucket    n  declining_rate
         Q1_lowest 7503        0.376116
                Q2 7499        0.604614
                Q3 7498        0.625634
        Q4_highest 7500        0.562000


In [10]:
# Additional signal check: average search position

position_df = df[df["avg_position"] > 0].copy()

position_df["position_bucket"] = pd.qcut(
    position_df["avg_position"],
    q=4,
    labels=["Q1_best_position", "Q2", "Q3", "Q4_worst_position"]
)

position_audit = (
    position_df.groupby("position_bucket", observed=False)
    .agg(
        n=("audit_declining", "size"),
        declining_rate=("audit_declining", "mean")
    )
    .reset_index()
)

print("SIGNAL — AVG_POSITION")
print(position_audit.to_string(index=False))

SIGNAL — AVG_POSITION
  position_bucket    n  declining_rate
 Q1_best_position 7412        0.550594
               Q2 7076        0.583946
               Q3 7115        0.611806
Q4_worst_position 7192        0.512792


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.